# Homeownership, Debt, and the American Dream

**Team:** Raghav Dewangan, Kartik Gopalani, Kavyan Patel, Rohan Nigam

**Project Proposal**

**Research Question:**
To what extent does the total amount of debt held by a U.S. citizen correlate with the age at which they first purchase a home?

This notebook will explore, analyze, and model this relationship using publicly available datasets.

## 1. Introduction
- Motivation and summary of proposal
- Importance of analyzing debt and homeownership timing

In the current 2026 economic climate, the American Dream of owning a home has shifted from a young adult's reality to an unbecoming fantasy later into adulthood. While high house prices are often blamed, the internal financial health of the average citizen, specifically their non-housing debt, is a persistent silent barrier.

This project is important because it moves beyond just looking at if people buy homes and instead focuses on when they will be able to. By quantifying the delay in acquiring homeownership caused by credit cards, auto loans, and student debt, we can better understand, explore, and visualize the relationship between personal debt, in whatever form that may be, and wealth milestones; in our case, homeownership.

**Research Question:** To what extent does the total amount of debt held by a U.S. citizen correlate with the age at which they first purchase a home?

- Overview of analysis approach

The ultimate goal is to determine whether there is an association between total personal debt and the timing of homeownership. If such a relationship exists, we aim to measure how strong the association is and estimate its magnitude. This leads into our analysis process. We will begin with Exploratory Data Analysis (EDA), using data visualization techniques such as scatter plots and box plots to identify potential patterns or clusters of individuals who purchase homes earlier versus those who experience longer delays. We will also perform data cleaning to address missing values and outliers that could distort the analysis and lead to unreliable results.

---


## 2. Data Acquisition

In [ ]:
import pandas as pd

In [ ]:
nls_duration = pd.read_excel("./data/nls/duration-of-employment-ages-18-to-36.xlsx")
nls_duration

#### NLSY97: Number of jobs held (ages 18–36)
This table captures job switching patterns and employment stability over early adulthood.

In [ ]:
nls_jobs = pd.read_excel('./data/nls/number-of-jobs-held-ages-18-to-36.xlsx')
nls_jobs

#### NLSY97: Partner status (ages 27 and 37)
This table provides household/relationship context that may influence homeownership timing.

In [ ]:
nls_partner_status = pd.read_excel('./data/nls/partner-status-at-ages-27-and-37.xlsx')
nls_partner_status

#### NLSY97: Percent weeks employed/unemployed/out of labor force (by education, ages 18–36)
This table captures labor-force attachment and unemployment exposure over time.

In [ ]:
nls_weeks_employment = pd.read_excel('./data/nls/percent-of-weeks-employment-by-education-ages-18-to-36.xlsx')
nls_weeks_employment

#### NLSY97: Training received (ages 18–36)
This table tracks skill-building/training exposure, which can affect earnings and home-buying readiness.

In [ ]:
nls_training = pd.read_excel('./data/nls/trainings-received-from-ages-18-to-36.xlsx')
nls_training

#### SCFP 2022 (Survey of Consumer Finances Public Data)
This dataset provides household-level financial characteristics (assets, debts, income) used to measure debt burden.

In [ ]:
scfp = pd.read_csv('./data/SCFP2022.csv')
scfp

#### USAUCSFRCONDOSMSAMID (FRED Condo Price Index)
This monthly U.S. condo price series is a market-condition control that helps separate debt effects from housing market effects.

In [ ]:
usa_condo_index = pd.read_csv('./data/USAUCSFRCONDOSMSAMID.csv')
usa_condo_index.head()

In [ ]:
datasets = {
    'nls_duration': nls_duration,
    'nls_jobs': nls_jobs,
    'nls_partner_status': nls_partner_status,
    'nls_weeks_employment': nls_weeks_employment,
    'nls_training': nls_training,
    'scfp': scfp,
    'usa_condo_index': usa_condo_index,
}

for name, d in datasets.items():
    print(f'\n=== {name} ===')
    print('shape:', d.shape)
    print('columns:', d.columns.tolist()[:15])

In [ ]:
from pathlib import Path

nls_paths = sorted(Path('./data/nls').glob('*.xlsx'))
for p in nls_paths:
    print(f'\n=== {p.name} ===')
    xls = pd.ExcelFile(p)
    print('sheets:', xls.sheet_names)
    raw = pd.read_excel(p, sheet_name=xls.sheet_names[0], header=None)
    print(raw.head(8).to_string(index=False, header=False))

In [ ]:
from pathlib import Path

nls_paths = sorted(Path('./data/nls').glob('*.xlsx'))
for p in nls_paths:
    xls = pd.ExcelFile(p)
    raw = pd.read_excel(p, sheet_name=xls.sheet_names[0], header=None, nrows=3)
    print(f"{p.name} | sheet={xls.sheet_names[0]} | A1={raw.iloc[0,0]}")

---


### Standardized NLS Loader

This function reads NLS publication-style XLSX tables, detects a likely header row, removes footnote/title artifacts, and returns:
- a cleaned wide table, and
- a tidy long table (`group`, `measure`, `value`).

In [ ]:
import re
from pathlib import Path
import pandas as pd


def _score_header_row(row_values):
    vals = [str(v).strip() for v in row_values if pd.notna(v) and str(v).strip() != '']
    if not vals:
        return -1

    text = ' '.join(vals).lower()
    non_empty = len(vals)

    score = non_empty
    if 'table' in text:
        score -= 3
    if any(k in text for k in ['education', 'age', 'sex', 'race', 'ethnicity', 'hispanic', 'status', 'total']):
        score += 3
    if any('unnamed' in v.lower() for v in vals):
        score -= 1
    return score


def _is_mostly_numeric_header(header_series):
    vals = [str(v).strip() for v in header_series.tolist() if str(v).strip() != '']
    if not vals:
        return True
    numeric_count = 0
    for v in vals:
        try:
            float(v)
            numeric_count += 1
        except Exception:
            pass
    return numeric_count / max(1, len(vals)) >= 0.5


def load_nls_table(path, sheet_name=0, header_row='auto'):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    raw = raw.dropna(axis=0, how='all').dropna(axis=1, how='all').reset_index(drop=True)

    if header_row == 'auto':
        top = min(20, len(raw))
        scored = [(_score_header_row(raw.iloc[i].tolist()), i) for i in range(top)]
        header_idx = max(scored, key=lambda x: x[0])[1]
    else:
        header_idx = int(header_row)

    header = raw.iloc[header_idx].astype(str).str.strip().replace({'nan': ''})

    if _is_mostly_numeric_header(header) and header_idx > 0:
        candidate = raw.iloc[header_idx - 1].astype(str).str.strip().replace({'nan': ''})
        if not _is_mostly_numeric_header(candidate):
            header = candidate
            header_idx = header_idx - 1

    header = [h if h != '' else f'col_{i}' for i, h in enumerate(header)]

    data = raw.iloc[header_idx + 1:].copy()
    data.columns = header
    data = data.dropna(axis=0, how='all').reset_index(drop=True)

    first_col = data.columns[0]
    data = data.rename(columns={first_col: 'group'})

    def clean_group(x):
        s = str(x)
        s = re.sub(r'\s+', ' ', s).strip()
        s = re.sub(r'[\.·…]+$', '', s).strip()
        return s

    data['group'] = data['group'].map(clean_group)

    long_df = data.melt(id_vars=['group'], var_name='measure', value_name='value')
    long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')
    long_df = long_df.dropna(subset=['value']).reset_index(drop=True)

    return data, long_df


nls_file_map = {
    'duration': './data/nls/duration-of-employment-ages-18-to-36.xlsx',
    'jobs': './data/nls/number-of-jobs-held-ages-18-to-36.xlsx',
    'partner_status': './data/nls/partner-status-at-ages-27-and-37.xlsx',
    'weeks_employment': './data/nls/percent-of-weeks-employment-by-education-ages-18-to-36.xlsx',
    'training': './data/nls/trainings-received-from-ages-18-to-36.xlsx',
}

nls_tables_wide = {}
nls_tables_long = {}

for key, filepath in nls_file_map.items():
    wide_df, long_df = load_nls_table(filepath)
    nls_tables_wide[key] = wide_df
    nls_tables_long[key] = long_df
    print(f"{key:16s} wide={wide_df.shape} | long={long_df.shape}")

nls_tables_long['jobs'].head()

### Understanding the NLSY97 Data in Context of Our Research Question

The National Longitudinal Survey of Youth 1997 (NLSY97) follows the same cohort of individuals from ages 18 to 36, providing a unique longitudinal perspective on how life circumstances evolve over time. While these summary tables don't contain individual-level debt and homeownership data, they provide crucial **contextual factors** that help us understand the broader economic environment in which debt and homeownership decisions are made.

**Why these factors matter for our research question:**
- **Employment stability** (duration, number of jobs) affects income and ability to service debt
- **Training and education** influence earning potential and debt capacity
- **Partner status** impacts household income and financial decision-making
- These factors help us control for confounding variables when analyzing the debt-homeownership relationship

### NLS Table Display and Visualization (EDA)

These cells show each cleaned NLS table and create comparable plots across all NLS datasets.

**Note on `Total 2`:** In these NLS tables, `Total` is the overall estimate across all individuals/groups in the table (for the stated age range and period). The trailing `2` is a table footnote marker from the original BLS/NLS spreadsheet, not part of the metric name itself.

### Exploring NLSY97 Summary Tables: Employment and Economic Context

The cleaned NLS tables below show aggregated statistics across different demographic groups. These patterns help us understand:

1. **Employment Patterns**: How job stability varies by education, age, and demographics
2. **Economic Context**: The labor market conditions that influence both debt accumulation and home-buying capacity
3. **Demographic Variations**: How different groups experience employment, which may correlate with debt levels and homeownership timing

**Connection to our research question**: Individuals with more stable employment and higher education levels typically have:
- Higher incomes → better debt-to-income ratios
- More predictable cash flows → easier mortgage qualification
- Greater financial security → earlier homeownership

This contextual data helps us interpret whether debt delays homeownership, or if both are driven by underlying employment and economic factors.

In [ ]:
from IPython.display import display, Markdown

for key, table in nls_tables_wide.items():
    display(Markdown(f"#### {key.replace('_', ' ').title()} (Wide)"))
    display(table)

for key, table in nls_tables_long.items():
    display(Markdown(f"#### {key.replace('_', ' ').title()} (Long/Tidy)"))
    display(table.head(20))

### Visualizing Employment and Economic Patterns

The visualizations below show key patterns from the NLSY97 data. These bar charts help us identify:

- **Which demographic groups** experience different employment outcomes
- **How employment metrics vary** across education levels, age groups, and other characteristics
- **Potential confounding factors** we need to account for when analyzing debt and homeownership

**Key insight for our analysis**: If we observe that certain groups (e.g., lower education, more job switching) also have higher debt and later homeownership, we need to carefully distinguish whether:
- Debt directly causes delayed homeownership, OR
- Both debt and delayed homeownership are consequences of underlying employment/economic factors

This is why we'll use the SCF data (with actual debt measurements) combined with these contextual NLS patterns to build a more robust model.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
axes = axes.flatten()

for i, (key, long_df) in enumerate(nls_tables_long.items()):
    ax = axes[i]

    measures = [m for m in long_df['measure'].dropna().unique().tolist() if str(m).strip() != '']
    chosen_measure = 'Total 2' if 'Total 2' in measures else measures[0]

    plot_df = long_df[long_df['measure'] == chosen_measure].copy()
    plot_df = plot_df[~plot_df['group'].str.contains('total', case=False, na=False)]
    plot_df = plot_df.sort_values('value', ascending=False).head(12)

    ax.barh(plot_df['group'], plot_df['value'])
    ax.invert_yaxis()
    ax.set_title(f"{key.replace('_', ' ').title()} ({chosen_measure})")
    ax.set_xlabel('Value')
    ax.set_ylabel('Group')

if len(nls_tables_long) < len(axes):
    for j in range(len(nls_tables_long), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### NLS EDA — Variable Explanations & Insights

The NLSY97 (National Longitudinal Survey of Youth 1997) tracks the same individuals from adolescence through adulthood, capturing life-course labor and family outcomes. The five summary tables used here each measure a distinct dimension of early-adult economic development, all potentially linked to when — and whether — someone can afford to buy a home.

---

**Duration of Employment** measures how long, on average, individuals stayed in a single job (in weeks) across ages 18–36. Longer job tenure signals employment stability, which matters for mortgage qualification: lenders typically require 2+ years of continuous employment history. The bar charts show that individuals with higher education and income tend to hold jobs longer, suggesting greater financial readiness for homeownership.

---

**Number of Jobs Held** counts how many distinct employers an individual worked for between ages 18–36. High job-switching in early adulthood is common but delays credit-building and savings accumulation. The EDA reveals that younger age cohorts (18–23) hold significantly more jobs on average (~5.1) compared to older cohorts (30–36, ~2.5), consistent with labor market churning before career stabilization — a period when debt accrues but incomes remain volatile.

---

**Partner Status** captures whether respondents were single, cohabiting, or married at ages 27 and 37 — two key life-stage checkpoints. Partnership status is a major homeownership predictor because dual-income households have higher combined borrowing capacity and are more likely to purchase property. The charts show a clear shift toward partnership by age 37, which aligns with the well-documented marriage-homeownership nexus.

---

**Percent Weeks Employed / Unemployed / Out of Labor Force** breaks down what fraction of each year respondents spent in each labor-force state, segmented by education level across ages 18–36. This captures earnings continuity: even brief unemployment spells can impair savings and increase debt reliance. The EDA shows lower-educated respondents spend meaningfully more weeks outside employment, reinforcing why education is a significant predictor of homebuying age in our SCF models.

---

**Training Received** records participation in formal or informal job training programs between ages 18–36. Training is a proxy for human capital investment — it often precedes wage growth and career advancement, reducing the debt-to-income burden over time. The charts suggest training participation varies substantially by education group, with less-educated respondents both needing training more and receiving it less consistently.

---

> **Connection to Research Question:** Collectively, these NLS dimensions paint a picture of early-adult economic instability — frequent job changes, gaps in employment, delayed partnership — that prolongs the period in which debt accumulates relative to income, ultimately pushing the age of first home purchase later. This provides qualitative support for the negative relationship between debt and homebuying age identified in the SCF regression analysis.

### Transitioning to Core Analysis Data

Now that we've established the economic and employment context from NLSY97, we turn to the **Survey of Consumer Finances (SCF)**, which provides the direct measurements we need:

- **Total debt amounts** (credit cards, auto loans, student debt, etc.)
- **Homeownership status and timing**
- **Income and assets**
- **Demographic characteristics**

This is where we'll perform the core EDA to answer our research question: *To what extent does total debt correlate with the age at which someone first purchases a home?*

**Our EDA approach** (as outlined in the proposal):
1. **Summary statistics** to understand the distribution of debt and homeownership ages
2. **Missing value analysis** to ensure data quality
3. **Outlier detection and handling** to prevent skewed results
4. **Visualizations**: scatter plots, box plots, and debt binning to identify patterns
5. **Debt-to-income ratios** to find potential tipping points where homeownership becomes less likely

### Analyzing Debt Composition (The "Debt Stack")
Total debt is a broad figure. To understand its impact on homeownership, we must distinguish between "good" debt (like student loans, which correlate with higher lifetime earnings) and "liquidity-draining" debt (like high-interest credit cards).

**Methodology**: We will decompose the DEBT variable into its primary components. By comparing the "Debt Stack" of homeowners versus non-homeowners in the prime first-time buyer age range (25–40), we can see if specific types of debt are more prevalent among those who remain in the rental market.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Define the possible debt columns based on SCF 2022 naming conventions
# 'EDN_INST' is the standard name for Education-related installment debt (Student Loans)
debt_map = {
    'CCDEBT': 'Credit Card',
    'EDN_INST': 'Student Loans',
    'INSTALL': 'Other Installment'
}

# 2. Check which columns actually exist in your 'scfp' dataframe
available_cols = [col for col in debt_map.keys() if col in scfp.columns]
print(f"Using available columns: {available_cols}")

# 3. Filter for the prime home-buying age (25-40)
# We use the full scfp here because your model_df might have already dropped these columns
prime_age_df = scfp[(scfp['AGE'] >= 25) & (scfp['AGE'] <= 40)].copy()

# 4. Calculate weighted means for each available debt type
# We divide the sum of (debt * weight) by the sum of weights
def weighted_mean(df, col):
    return (df[col] * df['WGT']).sum() / df['WGT'].sum()

stack_results = []
for own_status in [0, 1]:
    subset = prime_age_df[prime_age_df['OWN'] == own_status]
    for col in available_cols:
        stack_results.append({
            'Ownership': 'Homeowner' if own_status == 1 else 'Non-Homeowner',
            'Debt Type': debt_map[col],
            'Average Amount': weighted_mean(subset, col)
        })

stack_data = pd.DataFrame(stack_results)

# 5. Visualization
plt.figure(figsize=(10, 6))
sns.barplot(data=stack_data, x='Ownership', y='Average Amount', hue='Debt Type')

plt.title('Average Non-Housing Debt Composition (Ages 25-40)', fontsize=14)
plt.ylabel('Mean Weighted Debt Amount ($)', fontsize=12)
plt.xlabel('Ownership Status', fontsize=12)
plt.legend(title='Debt Category')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show() 


### Why we use these specific Markdowns:
1. **Weighted Means**: It's important to explain that we use WGT. The SCF is not a random sample; some households represent thousands of others. If we didn't use weights, our "averages" would be biased.

2. **The 25-40 Age Filter**: We specify this range because looking at debt for a 70-year-old doesn't tell us much about the timing of a first home purchase. This age range captures the "entry-level" market.

3. **Non-Housing Debt**: By excluding the mortgage itself, we are looking specifically at the "financial baggage" people carry into the home-buying process.

### Finding the DTI "Tipping Point"
Lenders primarily use the Debt-to-Income (DTI) ratio to determine mortgage eligibility. Even if a person has high income, a high DTI can disqualify them from a loan.

**Analysis Goal**:
Identify the "Tipping Point" which is the specific DTI level where the probability of homeownership begins to decline sharply. This helps validate the "threshold" theory mentioned in our proposal.

In [ ]:
# Create DTI Deciles (excluding those with 0 debt for a clearer trend)
debt_only = scfp[scfp['DEBT2INC'] > 0].copy()
debt_only['dti_decile'] = pd.qcut(debt_only['DEBT2INC'], 10, labels=False)

# Calculate ownership rate per decile
dti_trend = debt_only.groupby('dti_decile').agg({
    'OWN': 'mean',
    'DEBT2INC': 'median'
}).reset_index()

# Plot the trend
plt.figure(figsize=(10, 5))
sns.lineplot(data=dti_trend, x='DEBT2INC', y='OWN', marker='o', color='red')
plt.title('Homeownership Rate vs. Debt-to-Income Ratio')
plt.xlabel('Median DTI Ratio in Decile')
plt.ylabel('Homeownership Probability')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### Wealth vs. Debt: The "Liquidity Trap"
Homeownership requires a down payment. High debt often prevents the accumulation of liquid assets (cash/savings). We use a joint distribution plot to see the "cluster" of homeowners versus renters.

**Visualization Insight**:
In the plot below, we look for a "Dead Zone"—a region where debt is high and net worth is low to see if homeownership is virtually non-existent there, regardless of age.

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle

# Sample the data (weighted) for a cleaner scatter plot
sample_df = scfp.sample(2000, weights=scfp['WGT'], random_state=42).copy()

# Log-transform — clip NETWORTH at 0 so negatives don't produce NaN
sample_df['log_networth'] = np.log1p(sample_df['NETWORTH'].clip(lower=0))
sample_df['log_debt']     = np.log1p(sample_df['DEBT'])

g = sns.jointplot(data=sample_df, x='log_debt', y='log_networth',
                  hue='OWN', alpha=0.45, palette='viridis', height=7)
g.fig.suptitle("The Liquidity Trap: Debt vs. Net Worth by Ownership", y=1.02, fontsize=13)
g.ax_joint.set_xlabel("log(1 + Debt)  [higher = more debt]", fontsize=11)
g.ax_joint.set_ylabel("log(1 + Net Worth)  [higher = wealthier]", fontsize=11)

# ── Dead Zone annotation ──────────────────────────────────────────────────────
# High debt  : log1p > 10  ≈ debt > $22 000
# Low wealth : log1p < 10  ≈ net worth < $22 000
dz_x0, dz_y0 = 10.0, 0.0
dz_x1, dz_y1 = 16.5, 10.0   # 16.5 covers right edge; y=10 is the ceiling

rect = Rectangle((dz_x0, dz_y0), dz_x1 - dz_x0, dz_y1 - dz_y0,
                 linewidth=1.8, edgecolor='crimson', facecolor='crimson',
                 alpha=0.13, zorder=4)
g.ax_joint.add_patch(rect)
g.ax_joint.text(dz_x0 + 0.15, dz_y0 + 0.35,
                "⚠  Dead Zone\n(High Debt +\nLow Net Worth)",
                color='crimson', fontsize=9, fontweight='bold', va='bottom')

# Reference lines that define the zone boundary
g.ax_joint.axvline(dz_x0, color='crimson', linestyle='--', linewidth=1.0, alpha=0.6)
g.ax_joint.axhline(dz_y1, color='crimson', linestyle='--', linewidth=1.0, alpha=0.6)

plt.tight_layout()
plt.show()

# ── Compute what fraction of Dead Zone residents are owners ───────────────────
in_dz = sample_df[(sample_df['log_debt'] > dz_x0) & (sample_df['log_networth'] < dz_y1)]
out_dz = sample_df[~((sample_df['log_debt'] > dz_x0) & (sample_df['log_networth'] < dz_y1))]
dz_own_rate  = in_dz['OWN'].mean()
out_own_rate = out_dz['OWN'].mean()
print(f"Dead Zone   — n={len(in_dz):4d}  |  ownership rate: {dz_own_rate:.1%}")
print(f"Outside DZ  — n={len(out_dz):4d}  |  ownership rate: {out_own_rate:.1%}")
print(f"Ownership gap: {out_own_rate - dz_own_rate:.1%} lower inside the Dead Zone")


#### Interpreting the Dead Zone

The jointplot shows **two marginal histograms** (one per axis) plus the central scatter where each layer adds a different lens on the same data.

**How to Interpret the axes**
- **X-axis — log(1 + Debt):** a value of 10 = ~\$22 K in debt; 14 = ~\$1.2 M. Because debt distributions are heavily right-skewed, the log scale spreads observations out and makes the gradient more visible to viewers
- **Y-axis — log(1 + Net Worth):** 0 means literally zero (or negative, clipped) net worth; 14 = ~\$1.2 M in assets net of liabilities. The vertical spread separates the "asset-rich" top from the "asset-poor" bottom.
- **Color:** yellow/light = homeowners (`OWN=1`); purple/dark = renters (`OWN=0`).

**The Dead Zone (shaded red region)**

The crimson rectangle marks the combination of **high debt and low net worth**. Here everal things stand out:

1. **Near zero homeownership density.** Almost all points in the Dead Zone are purple (renters). This is the observed "liquidity trap" : a household carrying heavy debt obligations while holding little wealth simply cannot accumulate the down payment (~3–20 % of home value) required to enter the mortgage market.

2. **Why the vertical dashed line matters.** At log-debt ≈ 10 (~\$22 K), the color composition shifts. Below this threshold renters and owners collide; above it owners who appear are predominantly also high net worth (upper right), not low net worth. The line marks the point where debt load starts acting as a filter.

3. **Why the horizontal dashed line matters.** At log-net-worth ≈ 10 (~\$22 K), households below this level rarely own homes regardless of debt. This is consistent with the typical minimum cash-to-close requirement for a median-priced home in 2022. It shows the **asset floor** required to enter homeownership.

4. **Marginal histograms.** The top marginal (debt distribution) shows that owners and renters overlap heavily at zero-to-moderate debt, but renters dominate the right tail. The right marginal (net worth) shows owners stacked at higher values, confirming the selection: you must have *both* manageable debt *and* sufficient assets.

**Connection to the research question**

This visual directly supports the hypothesis that debt delays homeownership beyond a pure timing effect. The Dead Zone households are not just *younger* — they are structurally locked out of homeownership *at any age* given their balance-sheet position. Debt-to-income alone understates this barrier; the joint (debt, net-worth) space reveals the full constraint.


### Job Volatility and Timing (NLSY97)
Using the longitudinal NLSY97 data, we can look at "career stability." If an individual switches jobs frequently, they may struggle to show the "two years of steady employment" required by many mortgage lenders.

**Analysis Goal**:
Compare the number of jobs held by age 30 to the homeownership status at age 30.

In [ ]:
jobs_df = nls_tables_long['jobs'].copy()

# Filter out the "Total" rows to see demographic differences
# and focus on the 'Total 2' measure (which typically represents the full cohort)
demo_jobs = jobs_df[
    (~jobs_df['group'].str.contains('Total', case=False)) & 
    (jobs_df['measure'] == 'Total 2')
].sort_values('value', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=demo_jobs, x='value', y='group', palette='viridis')

plt.title('Average Number of Jobs Held from Ages 18 to 36', fontsize=14)
plt.xlabel('Mean Number of Jobs Held', fontsize=12)
plt.ylabel('Demographic Group', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

#### Reading the Bar Chart: Jobs Held by Demographic Group (Ages 18–36)

**What each bar represents**

Each bar shows the **mean number of jobs** held by individuals in that demographic group between ages 18 and 36, as tracked by the NLSY97 cohort. A higher bar means people in that group cycled through more employers over that ~18 year window.

---


**Key insights from the chart**

1. **High variation across groups.** The range across demographic groups spans roughly 1–2 jobs, which is substantial for an 18 year window. Groups at the top of the chart experience more employer turnover, which directly matters for mortgage qualification (lenders typically requiring 2 years of continuous employment history).

2. **Groups with more job switching face a structural delay.** Each additional job transition involves weeks to months of employment gaps, probationary periods at new employers, and potential income volatility, all of which reduce a lender's willingness to extend a mortgage. This is the mechanism linking job instability → delayed homeownership.

3. **Connection to debt timing.** The NLSY97 captured this same cohort during the ages at which they were also accumulating student loans and early-career debt. Groups with higher job volatility are also more likely to have intermittent income, making it harder to service debt, compounding the barrier that shows up in the SCF Dead Zone.

> **Takeaway:** High job-switching rates indicate career instability during the exact years when households should be building the savings and credit history needed for a down payment. This provides independent, longitudinal evidence that the debt delay relationship in our SCF model is not purely age driven, but it reflects a broader pattern of financial instability in early adulthood.


---


### Comprehensive Modeling Workflow (SCF-centered)

Because SCF is cross sectional, we cannot directly observe each person's exact age at first home purchase. We therefore use a **two-model strategy** that still answers the proposal question in a defensible way:

1. **Owner-age model (weighted OLS):** among homeowners (`OWN == 1`), estimate how debt burden is associated with current age while controlling for income, education, race, sex, and net worth.
2. **Ownership-timing proxy model (weighted logistic):** on the full sample, predict `OWN` from age, debt burden, and controls. Then estimate the age where predicted ownership probability reaches 50% at low vs high debt levels.

This yields both:
- effect-size estimates (association strength), and
- an interpretable timing variable (estimated delay in ownership age, in years).

We use SCF survey weights (`WGT`) and robust inference where applicable.

#### Step 1 — Build the Modeling Dataset

At a high level, we first construct a clean analysis table from SCF that includes our outcome variables, debt measures, controls, and survey weights.

This step ensures all downstream models are trained on the same filtered sample and consistently engineered features (for example, log-transformed debt/income/net worth).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Ensure SCF data is available in a fresh kernel
if 'scfp' not in globals():
    scfp = pd.read_csv('./data/SCFP2022.csv')

# Build SCF modeling table
scf_model_cols = [
    'AGE', 'OWN', 'DEBT', 'DEBT2INC', 'INCOME', 'EDUC', 'RACE', 'HHSEX', 'NETWORTH', 'WGT'
 ]

model_df = scfp[scf_model_cols].copy()

# Numeric conversion + basic cleaning
for c in scf_model_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors='coerce')

# Keep valid rows for core variables
model_df = model_df.dropna(subset=['AGE', 'OWN', 'DEBT', 'DEBT2INC', 'INCOME', 'WGT']).copy()
model_df = model_df[(model_df['WGT'] > 0) & (model_df['AGE'] >= 18)].copy()

# Feature engineering
model_df['log_debt'] = np.log1p(np.clip(model_df['DEBT'], a_min=0, a_max=None))
model_df['log_income'] = np.log1p(np.clip(model_df['INCOME'], a_min=0, a_max=None))
model_df['log_networth'] = np.log1p(np.clip(model_df['NETWORTH'], a_min=0, a_max=None))

# Owner subset for age model
owners_df = model_df[model_df['OWN'] == 1].copy()

print('Model table shape:', model_df.shape)
print('Owner subset shape:', owners_df.shape)
model_df[['AGE','OWN','DEBT','DEBT2INC','INCOME','WGT','log_debt']].head()

#### Step 1a — Assumptions and Diagnostic Checks

Before fitting the models, we should state what makes the inference reasonable.

**High-level assumptions behind the modeling section**

1. **Weighted observations are informative.** We use SCF survey weights so the fitted relationships better reflect the household population rather than the raw sample mix.
2. **The functional form is an approximation.** Log-transforming debt, income, and net worth is meant to make extremely skewed financial variables more stable and more suitable for linear/logistic modeling.
3. **The main confounders are at least partially controlled for.** Income, net worth, education, race, and sex are included because they are related to both debt burden and ownership outcomes.
4. **Multicollinearity is not severe enough to make coefficients meaningless.** If predictors move too closely together, coefficient estimates become unstable, so we check this directly below.
5. **For OLS inference, residual behavior should be broadly reasonable.** We will later inspect residual plots and a Q-Q plot after fitting the weighted OLS model.

**What we can check empirically in the notebook**
- a **correlation heatmap** among numeric predictors,
- **VIF values** to assess multicollinearity,
- and **OLS residual diagnostics** after the model is fit.

These checks do not prove the model is perfect, but they do show whether the assumptions are reasonable enough for a careful high-level inference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Numeric predictors used most directly in the SCF models
assumption_df = model_df[['DEBT2INC', 'log_debt', 'log_income', 'log_networth']].dropna().copy()

# --- Correlation heatmap ---
plt.figure(figsize=(7, 5))
sns.heatmap(assumption_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Predictor Correlation Heatmap')
plt.tight_layout()
plt.show()

# --- Variance Inflation Factors (VIF) ---
X_vif = sm.add_constant(assumption_df)
vif_table = pd.DataFrame({
    'variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

print('Rule of thumb: VIF < 5 is usually comfortable; VIF > 10 suggests serious multicollinearity.')
display(vif_table)


#### Step 1b — Variable Selection (Backward Elimination)

To keep the model significant and transparent, we run backward variable selection on the weighted OLS specification.

At a high level, we start with the full set of candidate terms, iteratively remove the least useful term (highest non-significant p-value), and stop once all remaining removable terms are significant at $\alpha = 0.05$.

> We keep `log_debt` in the model throughout, since it is the primary variable tied directly to the research question.

In [ ]:
# Backward variable selection for weighted OLS (keeping log_debt fixed)
alpha_remove = 0.05
forced_terms = {'log_debt'}

candidate_terms = [
    'log_debt',
    'DEBT2INC',
    'log_income',
    'log_networth',
    'C(EDUC)',
    'C(RACE)',
    'C(HHSEX)'
]

def make_formula(terms):
    return 'AGE ~ ' + ' + '.join(terms)

current_terms = candidate_terms.copy()
selection_steps = []

while True:
    current_formula = make_formula(current_terms)
    current_fit = smf.wls(
        current_formula,
        data=owners_df,
        weights=owners_df['WGT']
).fit(cov_type='HC3')

    term_table = current_fit.wald_test_terms(skip_single=False).summary_frame()
    p_col = 'P>chi2' if 'P>chi2' in term_table.columns else term_table.columns[-1]
    term_pvals = term_table[p_col].astype(float)

    removable_terms = [t for t in current_terms if t not in forced_terms]
    removable_pvals = term_pvals.loc[term_pvals.index.intersection(removable_terms)]

    if removable_pvals.empty:
        break

    worst_term = removable_pvals.idxmax()
    worst_p = float(removable_pvals.max())

    selection_steps.append({
        'formula': current_formula,
        'removed_term': worst_term if worst_p > alpha_remove else None,
        'max_removable_p': worst_p,
    })

    if worst_p <= alpha_remove:
        break

    current_terms.remove(worst_term)

selected_ols_terms = current_terms.copy()
selected_ols_formula = make_formula(selected_ols_terms)

selection_trace = pd.DataFrame(selection_steps)
print('Selected OLS formula:', selected_ols_formula)
display(selection_trace.tail(10))

#### Step 2 — Estimate the Core Association (Weighted OLS)

Next, we fit a weighted linear model on homeowners only to estimate how debt burden is associated with homeowner age after adjusting for key confounders.

This provides an interpretable coefficient-level view (direction, size, and significance) of the debt–age relationship.

In [ ]:
# 1) Weighted OLS among homeowners: association between debt burden and age
full_ols_formula = "AGE ~ log_debt + DEBT2INC + log_income + log_networth + C(EDUC) + C(RACE) + C(HHSEX)"
ols_formula = selected_ols_formula if 'selected_ols_formula' in globals() else full_ols_formula

print('Using OLS formula:', ols_formula)

ols_fit = smf.wls(
    ols_formula,
    data=owners_df,
    weights=owners_df['WGT']
).fit(cov_type='HC3')

print(ols_fit.summary())

# Key coefficients table
ols_key = ols_fit.params.to_frame('coef').join(
    ols_fit.bse.to_frame('std_err')
).join(
    ols_fit.pvalues.to_frame('p_value')
)
ols_key.loc[[idx for idx in ols_key.index if idx in ['log_debt', 'DEBT2INC', 'log_income', 'log_networth']]]

In [ ]:
from statsmodels.graphics.gofplots import qqplot

# OLS diagnostic plots: fitted vs residuals and normal Q-Q
residuals = ols_fit.resid
fitted = ols_fit.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(fitted, residuals, alpha=0.25)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('OLS Residuals vs Fitted')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')

qqplot(residuals, line='45', ax=axes[1], fit=True)
axes[1].set_title('OLS Residual Q-Q Plot')

plt.tight_layout()
plt.show()


**How to read these assumption checks.**

- In the **correlation heatmap**, we mainly want to avoid predictors that are almost perfectly correlated with one another.
- In the **VIF table**, values below about **5** are usually comfortable, while values above **10** are a warning sign for severe multicollinearity.
- In the **residuals vs fitted** plot, a roughly patternless cloud around zero is preferable to a strong curve or funnel shape.
- In the **Q-Q plot**, points close to the 45° line suggest residuals are reasonably close to normal. Exact normality is not required here because we use robust standard errors, but extreme departures would still weaken the classical OLS interpretation.

These diagnostics are meant to justify that the models are a reasonable approximation for inference, not to claim that every assumption holds perfectly.

#### Step 2b — Formal Hypothesis Test

To connect the model directly back to the research question, we perform a formal test on the `log_debt` coefficient in the weighted OLS model.

At a high level, this asks whether debt burden has a statistically detectable association with homeowner age after controlling for income, education, race, sex, and net worth.

We test:
- $H_0$: $\beta_{\log(\mathrm{debt})} = 0$
- $H_A$: $\beta_{\log(\mathrm{debt})} \neq 0$

If the p-value is below $\alpha = 0.05$, we reject the null hypothesis and conclude that debt burden is significantly associated with age in this model.

In [ ]:
# Formal hypothesis test for the debt coefficient in the weighted OLS model
alpha = 0.05
test_term = 'log_debt'

beta_hat = float(ols_fit.params[test_term])
std_err = float(ols_fit.bse[test_term])
test_stat = float(ols_fit.tvalues[test_term])
p_value = float(ols_fit.pvalues[test_term])
ci_low, ci_high = [float(x) for x in ols_fit.conf_int().loc[test_term]]
reject_null = p_value < alpha

hypothesis_test_summary = pd.DataFrame({
    'term': [test_term],
    'estimate': [beta_hat],
    'std_err': [std_err],
    'test_statistic': [test_stat],
    'p_value': [p_value],
    'ci_lower_95': [ci_low],
    'ci_upper_95': [ci_high],
    'alpha': [alpha],
    'reject_null': [reject_null],
})

decision_text = 'Reject $H_0$' if reject_null else 'Fail to reject $H_0$'
interpretation_text = (
    f"Debt burden is {'significantly' if reject_null else 'not significantly'} associated with homeowner age "
    f"in the weighted OLS model (estimate={beta_hat:.3f}, test statistic={test_stat:.3f}, "
    f"p-value={p_value:.3e}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}])."
 )

print(decision_text)
print(interpretation_text)
hypothesis_test_summary

**Interpretation note.** This test gives formal inferential evidence that debt burden is associated with age **within the homeowner sample**, after controlling for key financial and demographic variables. Because SCF is cross-sectional and does not directly record each respondent's age at first home purchase, this result should be interpreted as evidence of association rather than a causal estimate of how many years debt delays first-time homeownership. The logistic model and probability-gap analysis complement this test by describing how ownership likelihood varies across debt profiles.

#### Step 3 — Model Ownership Probability and Timing Proxy

We then train a weighted logistic model on the full sample to estimate homeownership probability across age and debt profiles.

At a high level, this lets us report model quality (cross-validated ROC-AUC/Brier) and translate predictions into an intuitive timing variable (or probability-gap fallback) for the research question.

In [ ]:
# 2) Weighted logistic model on full sample + CV + timing proxy
logit_df = model_df.copy()
logit_df['age_x_logdebt'] = logit_df['AGE'] * logit_df['log_debt']

feature_cols = ['AGE', 'log_debt', 'DEBT2INC', 'log_income', 'log_networth', 'age_x_logdebt', 'EDUC', 'RACE', 'HHSEX']
X = logit_df[feature_cols]
y = logit_df['OWN'].astype(int).values
w = logit_df['WGT'].values

num_cols = ['AGE', 'log_debt', 'DEBT2INC', 'log_income', 'log_networth', 'age_x_logdebt']
cat_cols = ['EDUC', 'RACE', 'HHSEX']

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
    ]
)

logit_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
try:
    cv_scores = cross_validate(
        logit_pipe, X, y, cv=cv, scoring=['roc_auc', 'neg_brier_score'],
        params={'clf__sample_weight': w}
    )
except TypeError:
    cv_scores = cross_validate(
        logit_pipe, X, y, cv=cv, scoring=['roc_auc', 'neg_brier_score'],
        fit_params={'clf__sample_weight': w}
    )

print('5-fold ROC-AUC:', np.mean(cv_scores['test_roc_auc']))
print('5-fold Brier score:', -np.mean(cv_scores['test_neg_brier_score']))

# Fit once on full data for interpretation and timing-proxy calculation
logit_pipe.fit(X, y, clf__sample_weight=w)

# Predicted ownership-probability at each age for chosen debt levels
debt_q25 = np.quantile(logit_df['log_debt'], 0.25)
debt_q75 = np.quantile(logit_df['log_debt'], 0.75)

template = {
    'DEBT2INC': np.median(logit_df['DEBT2INC']),
    'log_income': np.median(logit_df['log_income']),
    'log_networth': np.median(logit_df['log_networth']),
    'EDUC': int(np.round(logit_df['EDUC'].median())),
    'RACE': int(np.round(logit_df['RACE'].median())),
    'HHSEX': int(np.round(logit_df['HHSEX'].median())),
}

def ownership_curve(log_debt_level):
    ages = np.arange(20, 71)
    grid = pd.DataFrame({
        'AGE': ages,
        'log_debt': log_debt_level,
        'DEBT2INC': template['DEBT2INC'],
        'log_income': template['log_income'],
        'log_networth': template['log_networth'],
        'EDUC': template['EDUC'],
        'RACE': template['RACE'],
        'HHSEX': template['HHSEX'],
    })
    grid['age_x_logdebt'] = grid['AGE'] * grid['log_debt']
    p = logit_pipe.predict_proba(grid[feature_cols])[:, 1]
    return ages, p

def crossing_age(ages, probs, target):
    diff = probs - target
    idx = np.where(np.diff(np.sign(diff)) != 0)[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    x0, x1 = ages[i], ages[i + 1]
    y0, y1 = probs[i], probs[i + 1]
    if y1 == y0:
        return float(x1)
    return float(x0 + (target - y0) * (x1 - x0) / (y1 - y0))

ages_low, p_low = ownership_curve(debt_q25)
ages_high, p_high = ownership_curve(debt_q75)

# Timing proxy at reference age
ref_age = 35
ref_idx = np.where(ages_low == ref_age)[0][0]
p_low_ref = p_low[ref_idx]
p_high_ref = p_high[ref_idx]

eq_age_high_for_low = crossing_age(ages_high, p_high, p_low_ref)
eq_age_low_for_high = crossing_age(ages_low, p_low, p_high_ref)

shift_high_minus_low = eq_age_high_for_low - ref_age if not np.isnan(eq_age_high_for_low) else np.nan
shift_low_minus_high = eq_age_low_for_high - ref_age if not np.isnan(eq_age_low_for_high) else np.nan
prob_gap_at_ref_age = p_high_ref - p_low_ref

print(f'Predicted ownership at age {ref_age} (low debt):  {p_low_ref:.3f}')
print(f'Predicted ownership at age {ref_age} (high debt): {p_high_ref:.3f}')

if not np.isnan(shift_high_minus_low):
    print(f'Equivalent-age shift (high debt relative to low): {shift_high_minus_low:.2f} years')
elif not np.isnan(shift_low_minus_high):
    print(f'Equivalent-age shift (low debt relative to high): {shift_low_minus_high:.2f} years')
else:
    print('Equivalent-age shift not identifiable in [20,70]; reporting probability gap instead.')
    print(f'Ownership probability gap at age {ref_age} (high - low debt): {prob_gap_at_ref_age:.3f}')

pd.DataFrame({
    'age': ages_low,
    'p_own_low_debt_q25': p_low,
    'p_own_high_debt_q75': p_high,
}).head()

### Basic Interpretation of the Ownership-Probability Model

This cell is fitting a **weighted logistic regression** that predicts whether someone is a homeowner (`OWN = 1`) using age, debt burden, income, net worth, and demographics.

**What the output is telling us**

1. **The model has decent predictive power.**  
   The **ROC-AUC is about 0.778**, which means the model does a reasonably good job distinguishing homeowners from non-homeowners. It is not perfect, but it is clearly better than random guessing.

2. **The predictions are fairly well calibrated.**  
   The **Brier score is about 0.095**, and lower is better. This suggests the predicted probabilities are fairly close to the actual ownership outcomes.

3. **At age 35, the model predicts a higher ownership probability for the high-debt profile than the low-debt profile.**  
   Specifically, the model gives about **0.867** for the low-debt profile and **0.985** for the high-debt profile, a gap of about **0.118**.

4. **That result is important, but it should be interpreted carefully.**  
   This does **not** mean debt is helping people buy homes. In cross-sectional SCF data, people who already own homes often also carry **more total debt** because they may have mortgages, home-related loans, or larger balance sheets overall. So here, debt can partly act as a marker of being financially established enough to already be in the homeowner group.

5. **Why no "equivalent-age shift" was found.**  
   The model could not find a clean age where the two debt profiles cross in a way that produces a stable timing-shift estimate between ages 20 and 70. So instead of saying "high debt delays ownership by X years," this output reports the **probability gap at a reference age**.

**High-level takeaway**
- the ownership model is **useful but DEFINETELY not causal**,
- debt is clearly related to ownership status in the data,
- but in the full SCF sample, **higher debt can coincide with higher ownership probability** because existing homeowners often hold larger liabilities as well.

So this cell should be read as a **prediction / classification result**, not as direct proof that more debt causes earlier homeownership. That is why it complements the weighted OLS and the other contextual analyses rather than replacing them.


### Model Visualization + Section 6 Summary Artifacts

The cells below create:
- an ownership-probability curve plot (low vs high debt), and
- a compact, publication-ready summary table of key model outputs for the Results section.

In [ ]:
import matplotlib.pyplot as plt

# ---- Visualization: ownership probability curves ----
curve_df = pd.DataFrame({
    'age': ages_low,
    'Low debt (Q25)': p_low,
    'High debt (Q75)': p_high,
})

plt.figure(figsize=(9, 5))
plt.plot(curve_df['age'], curve_df['Low debt (Q25)'], label='Low debt (Q25)', linewidth=2)
plt.plot(curve_df['age'], curve_df['High debt (Q75)'], label='High debt (Q75)', linewidth=2)
plt.axvline(ref_age, linestyle='--', linewidth=1, color='gray', label=f'Reference age = {ref_age}')
plt.ylim(0, 1)
plt.xlabel('Age')
plt.ylabel('Predicted probability of homeownership')
plt.title('Predicted Ownership Curves by Debt Burden (Weighted Logistic Model)')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# ---- Publication-ready summary table for Section 6 ----
coef_cols = ['coef', 'std_err', 'p_value']
ols_focus = ols_key.loc[[k for k in ['log_debt', 'DEBT2INC', 'log_income', 'log_networth'] if k in ols_key.index], coef_cols].copy()
ols_focus = ols_focus.rename_axis('term').reset_index()
ols_focus['model'] = 'Weighted OLS (owners only)'

summary_rows = [
    {'metric': 'Logistic 5-fold ROC-AUC', 'value': float(np.mean(cv_scores['test_roc_auc']))},
    {'metric': 'Logistic 5-fold Brier score', 'value': float(-np.mean(cv_scores['test_neg_brier_score']))},
    {'metric': f'Predicted OWN at age {ref_age} (low debt Q25)', 'value': float(p_low_ref)},
    {'metric': f'Predicted OWN at age {ref_age} (high debt Q75)', 'value': float(p_high_ref)},
    {'metric': f'Ownership probability gap at age {ref_age} (high-low)', 'value': float(prob_gap_at_ref_age)},
]

if not np.isnan(shift_high_minus_low):
    summary_rows.append({
        'metric': f'Equivalent-age shift: high debt relative to low (years)',
        'value': float(shift_high_minus_low),
    })
elif not np.isnan(shift_low_minus_high):
    summary_rows.append({
        'metric': f'Equivalent-age shift: low debt relative to high (years)',
        'value': float(shift_low_minus_high),
    })
else:
    summary_rows.append({
        'metric': 'Equivalent-age shift identifiable in [20,70]',
        'value': np.nan,
    })

logit_summary = pd.DataFrame(summary_rows)

results_table_section6 = {
    'ols_key_coefficients': ols_focus,
    'logit_summary_metrics': logit_summary,
}

print('Section 6 artifact created: results_table_section6')
print('\nOLS coefficient summary:')
display(ols_focus)
print('\nLogistic/timing summary:')
display(logit_summary)

To broaden the analysis beyond SCF-only modeling, we build a small synthesis layer across datasets:

1. **SCF (microdata):** debt burden and ownership patterns by age bucket
2. **NLS (summary context):** employment stability indicators by age bucket
3. **FRED Condo Index:** housing market trend pressure over time

This section does not claim causal identification across datasets; instead, it provides aligned contextual evidence that helps interpret the debt-homeownership relationship more holistically.

In [ ]:
import re
from IPython.display import Markdown, display

# ---------- Helpers ----------
def _weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

def _normalize_age_label(text):
    t = re.sub(r'\s+', ' ', str(text).replace('\n', ' ')).strip()
    return t

def _age_bucket_from_number(age):
    if 18 <= age <= 23:
        return 'Ages 18 to 23'
    if 24 <= age <= 29:
        return 'Ages 24 to 29'
    if 30 <= age <= 36:
        return 'Ages 30 to 36'
    return 'Other'

age_buckets = ['Ages 18 to 23', 'Ages 24 to 29', 'Ages 30 to 36']

# ---------- A) SCF age-bucket summary ----------
scf_age = model_df.copy()
scf_age['age_bucket'] = scf_age['AGE'].astype(int).map(_age_bucket_from_number)
scf_age = scf_age[scf_age['age_bucket'] != 'Other'].copy()

scf_synth_rows = []
for bucket, g in scf_age.groupby('age_bucket', sort=False):
    scf_synth_rows.append({
        'age_bucket': bucket,
        'scf_ownership_rate': _weighted_mean(g['OWN'], g['WGT']),
        'scf_avg_log_debt': _weighted_mean(g['log_debt'], g['WGT']),
        'scf_avg_debt2inc': _weighted_mean(g['DEBT2INC'], g['WGT']),
        'scf_n_obs': int(len(g)),
    })

scf_age_summary = pd.DataFrame(scf_synth_rows)

# ---------- B) NLS context by age bucket ----------
nls_context = pd.DataFrame({'age_bucket': age_buckets})

# Pull 'jobs held' context from NLS jobs table (total row across population)
jobs_wide = nls_tables_wide['jobs'].copy()
jobs_wide['group_norm'] = jobs_wide['group'].astype(str).map(_normalize_age_label)
jobs_total = jobs_wide[jobs_wide['group_norm'].str.contains('total, ages 18 to 36', case=False, na=False)]
if jobs_total.empty:
    jobs_total = jobs_wide[jobs_wide['group_norm'].str.contains('total', case=False, na=False)]
if not jobs_total.empty:
    jobs_total = jobs_total.iloc[0]
    for col in jobs_wide.columns:
        col_norm = _normalize_age_label(col)
        if col_norm in age_buckets:
            nls_context.loc[nls_context['age_bucket'] == col_norm, 'nls_avg_jobs'] = pd.to_numeric(jobs_total[col], errors='coerce')

# Try employment status extraction from long table first
weeks_long = nls_tables_long['weeks_employment'].copy()
weeks_long['group_norm'] = weeks_long['group'].astype(str).str.lower().str.replace(r'\s+', ' ', regex=True)
weeks_long['measure_norm'] = weeks_long['measure'].astype(str).map(_normalize_age_label)

employed = weeks_long[
    weeks_long['group_norm'].str.contains('employed', na=False)
    & ~weeks_long['group_norm'].str.contains('unemployed|out of labor', na=False)
]
unemployed = weeks_long[weeks_long['group_norm'].str.contains('unemployed', na=False)]

for bucket in age_buckets:
    e_val = pd.to_numeric(employed.loc[employed['measure_norm'] == bucket, 'value'], errors='coerce').mean()
    u_val = pd.to_numeric(unemployed.loc[unemployed['measure_norm'] == bucket, 'value'], errors='coerce').mean()
    nls_context.loc[nls_context['age_bucket'] == bucket, 'nls_employed_pct'] = e_val
    nls_context.loc[nls_context['age_bucket'] == bucket, 'nls_unemployed_pct'] = u_val

# Fallback: if long extraction is empty/NaN, try wide table parsing
if nls_context[['nls_employed_pct', 'nls_unemployed_pct']].isna().all().all():
    weeks_wide = nls_tables_wide['weeks_employment'].copy()
    weeks_wide['group_norm'] = weeks_wide['group'].astype(str).str.lower().str.replace(r'\s+', ' ', regex=True)

    age_col_map = {}
    for col in weeks_wide.columns:
        c_norm = _normalize_age_label(col)
        if c_norm in age_buckets:
            age_col_map[c_norm] = col

    emp_rows = weeks_wide[
        weeks_wide['group_norm'].str.contains('employed', na=False)
        & ~weeks_wide['group_norm'].str.contains('unemployed|out of labor', na=False)
]
    unemp_rows = weeks_wide[weeks_wide['group_norm'].str.contains('unemployed', na=False)]

    if not emp_rows.empty and not unemp_rows.empty and age_col_map:
        emp_row = emp_rows.iloc[0]
        unemp_row = unemp_rows.iloc[0]
        for bucket, col in age_col_map.items():
            nls_context.loc[nls_context['age_bucket'] == bucket, 'nls_employed_pct'] = pd.to_numeric(emp_row[col], errors='coerce')
            nls_context.loc[nls_context['age_bucket'] == bucket, 'nls_unemployed_pct'] = pd.to_numeric(unemp_row[col], errors='coerce')

# ---------- C) Condo market trend summary ----------
condo = usa_condo_index.copy()
condo.columns = [str(c).strip() for c in condo.columns]

date_candidates = [c for c in condo.columns if 'date' in c.lower()]
date_col = date_candidates[0] if date_candidates else condo.columns[0]

value_candidates = [c for c in condo.columns if c != date_col]
if len(value_candidates) == 0:
    raise ValueError('Condo index dataset appears to have only one column; cannot identify value column.')
value_col = value_candidates[0]

condo[date_col] = pd.to_datetime(condo[date_col], errors='coerce')
condo[value_col] = pd.to_numeric(condo[value_col], errors='coerce')
condo = condo.dropna(subset=[date_col, value_col]).sort_values(date_col).copy()

start_val = float(condo.iloc[0][value_col])
end_val = float(condo.iloc[-1][value_col])
n_years = max((condo.iloc[-1][date_col] - condo.iloc[0][date_col]).days / 365.25, 1e-9)
long_run_cagr = (end_val / start_val) ** (1 / n_years) - 1

recent_cutoff = condo.iloc[-1][date_col] - pd.DateOffset(years=5)
condo_recent = condo[condo[date_col] >= recent_cutoff]
recent_start = float(condo_recent.iloc[0][value_col])
recent_end = float(condo_recent.iloc[-1][value_col])
recent_years = max((condo_recent.iloc[-1][date_col] - condo_recent.iloc[0][date_col]).days / 365.25, 1e-9)
recent_cagr = (recent_end / recent_start) ** (1 / recent_years) - 1

condo_summary = pd.DataFrame([
    {'metric': 'Condo index start', 'value': start_val},
    {'metric': 'Condo index latest', 'value': end_val},
    {'metric': 'Condo long-run CAGR', 'value': long_run_cagr},
    {'metric': 'Condo recent 5Y CAGR', 'value': recent_cagr},
])

# ---------- D) Cross-dataset synthesis table ----------
cross_dataset_synthesis = scf_age_summary.merge(nls_context, on='age_bucket', how='left')
cross_dataset_synthesis = cross_dataset_synthesis.sort_values('age_bucket')

display(Markdown('#### Cross-Dataset Synthesis Table'))
display(cross_dataset_synthesis)

display(Markdown('#### Condo Market Trend Summary'))
display(condo_summary)

# ---------- E) Analytical conclusions (concise) ----------
young = cross_dataset_synthesis[cross_dataset_synthesis['age_bucket'] == 'Ages 18 to 23']
older = cross_dataset_synthesis[cross_dataset_synthesis['age_bucket'] == 'Ages 30 to 36']

if not young.empty and not older.empty:
    own_gap = float(older['scf_ownership_rate'].iloc[0] - young['scf_ownership_rate'].iloc[0])
    dti_gap = float(young['scf_avg_debt2inc'].iloc[0] - older['scf_avg_debt2inc'].iloc[0])
else:
    own_gap = np.nan
    dti_gap = np.nan

conclusion_md = f"""
#### Cross-Dataset Analytical Conclusions
- SCF shows an ownership-rate difference of **{own_gap:.3f}** between ages 30–36 and 18–23, while debt-to-income burden differs by **{dti_gap:.3f}** (18–23 minus 30–36).
- NLS context indicates that employment/job patterns vary across these same age buckets, suggesting labor-market conditions likely co-move with debt burden and ownership readiness.
- The condo index shows sustained market pressure (long-run CAGR ≈ **{long_run_cagr:.2%}**, recent 5Y CAGR ≈ **{recent_cagr:.2%}**), which supports interpreting debt effects within a broader affordability environment.
- Together, these datasets provide converging evidence: household debt burden, employment context, and market prices jointly align with delayed or reduced homeownership readiness among younger cohorts.
"""
display(Markdown(conclusion_md))

# Keep artifacts for reporting
cross_dataset_artifacts = {
    'cross_dataset_synthesis': cross_dataset_synthesis,
    'condo_summary': condo_summary,
}
if 'results_table_section6' in globals():
    results_table_section6['cross_dataset_synthesis'] = cross_dataset_synthesis
    results_table_section6['condo_summary'] = condo_summary

### Narrative Interpretation — Cross Dataset Context

Putting the three datasets together provides a clearer story:

- **SCF (core signal):** Ownership is higher in ages 30–36 than 18–23 by about **0.063**, while debt-to-income burden is higher for ages 18–23 by about **2.251** points (18–23 minus 30–36).
- **NLS (labor context):** Jobs-held patterns vary strongly by age (for example, higher cumulative job counts in younger groups), supporting the interpretation that labor-market trajectories and debt burdens move together with ownership readiness.
- **Condo index (market pressure):** National condo prices rose at roughly **4.28% long-run CAGR** and **5.84% recent 5Y CAGR**, indicating sustained affordability pressure in the broader market environment.

**Integrated takeaway:** The debt-homeownership relationship observed in SCF is most plausibly interpreted within a broader system where labor-market dynamics (NLS context) and rising housing costs (FRED condo index) jointly shape ownership timing and readiness.

## Results & Discussion
- Key findings, metrics (e.g., debt-to-income tipping point)
- Interpretation of results

In [ ]:
from IPython.display import Markdown, display

# Small auto-generated writeup snippet for Section 6
results_paragraph = (
    f"Using weighted SCF models, the logistic classifier achieved an average 5-fold ROC-AUC of "
    f"{np.mean(cv_scores['test_roc_auc']):.3f} and a Brier score of "
    f"{-np.mean(cv_scores['test_neg_brier_score']):.3f}. "
    f"At age {ref_age}, predicted homeownership probability is "
    f"{p_low_ref:.3f} for the low-debt (Q25) profile and {p_high_ref:.3f} for the high-debt (Q75) profile, "
    f"a gap of {prob_gap_at_ref_age:.3f}. "
    f"Our formal hypothesis test on the weighted OLS debt coefficient leads us to "
    f"{'reject' if reject_null else 'fail to reject'} the null hypothesis that the debt effect is zero "
    f"(estimate={beta_hat:.3f}, p={p_value:.2e}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]). "
    f"Overall, this provides evidence that debt burden is statistically associated with homeowner age in this sample, "
    f"while remaining an observational association rather than a causal estimate of delayed first-time homeownership."
 )

display(Markdown('**Results Summary:**'))
display(Markdown(results_paragraph))

### Narrative Interpretation — Final Results

At a high level, the modeling and synthesis results are consistent with the project goal:

1. **Predictive signal is meaningful:** The ownership model performs reasonably well (ROC-AUC ≈ **0.778**, Brier ≈ **0.095**).
2. **Debt association is statistically strong:** The formal test rejects $H_0$ for the debt coefficient in weighted OLS (estimate ≈ **-0.818**, p-value **< 0.001**, 95% CI roughly **[-0.885, -0.752]**).
3. **Context supports interpretation:** Cross-dataset evidence suggests debt patterns should be interpreted alongside labor-market pathways (NLS context) and housing-price pressure (condo trend).

**Bottom line:** We find robust evidence of an association between debt burden and homeownership-related outcomes in this sample, while appropriately avoiding causal claims due to cross-sectional design.

---


## 7. Conclusion

This project asked: **To what extent does total debt correlate with the age at which a U.S. citizen first purchases a home?** Across the full analysis pipeline, the evidence supports a clear conclusion: debt burden is meaningfully associated with homeownership-related outcomes, and that relationship is statistically robust in our sample.

Our weighted OLS analysis (with backward variable selection) retained debt burden as a key explanatory variable after controls, while the formal hypothesis test on the debt coefficient rejected the null hypothesis. In practical terms, this means the observed relationship is not likely due to random variation in the analyzed data. The ownership model also showed useful predictive signal (ROC-AUC and Brier performance), reinforcing that the selected features capture real structure in ownership outcomes.

Just as importantly, the project moved beyond single-dataset interpretation. SCF provided the core household-level debt and ownership signal; NLS provided labor-market and life-course context; and the condo price index captured broader affordability pressure. Taken together, these datasets tell a more complete story: debt burden, employment pathways, and housing market conditions appear to jointly shape ownership readiness, especially for younger cohorts.

In the broader policy and social context, this matters because the analysis reframes homeownership from a binary question (“whether”) to a timing and readiness question (“when”). That framing is more useful for understanding financial mobility, wealth building, and inequality over the life course. Interventions targeting debt pressure and early-career financial stability may have meaningful downstream effects on homeownership access.

At the same time, we keep interpretation disciplined: these are observational, largely cross-sectional associations, not causal proofs of delayed home purchase in exact years. Even with that limitation, the consistency across model-based evidence and cross-dataset context makes the findings substantively valuable and aligned with the original project proposal.

---


## 8. References
- https://www.federalreserve.gov/econres/scfindex.htm
- https://www.bls.gov/nls/nlsy97.htm
    - Methods for using NLSY - https://www.bls.gov/opub/hom/pdf/nls-20030904.pdf
- https://fred.stlouisfed.org/series/USAUCSFRCONDOSMSAMID
